# Lesson 21 — Spatial Branch-and-Bound

## Learning objectives

1. State the spatial-B&B algorithm for nonconvex MINLP.
2. Recognize the role of FBBT, OBBT, and convex relaxations inside the tree.
3. Read a `discopt` global-mode log: bounds, gap, node count.
4. Diagnose when spatial B&B is failing (gap not closing).

## 1. Algorithm

```
Stack = {root box B}
LB = -inf, UB = +inf, incumbent = None
while gap > tol and Stack non-empty:
    pick node N (with box B_N)
    LB_N = solve convex relaxation over B_N
    if relaxation infeasible:        continue   # prune (infeasibility)
    if LB_N >= UB - tol:              continue   # prune (bound)
    if relaxation looks "tight" — auxiliaries near McCormick-tight,
       integer variables near-integer:
        x_local = local_NLP(N)                  # try to recover a primal point
        if x_local feasible: UB <- min(UB, f(x_local))
    if (UB - LB_N) / max(1, |UB|) < tol: continue   # gap closed at this node
    pick branching variable j (most violated, longest edge, etc.)
    split B_N along j -> two children, push
update LB <- min LB_N over open nodes
```

This is the spatial B&B of {cite:p}`Smith1999,Tawarmalani2005,Belotti2009`. The key differences from MILP B&B (Lesson 6) are **branching on continuous variables** (typically the auxiliaries $w_{ij} = x_i x_j$ that linearise nonconvex terms) to refine the convex relaxation, and the `relaxation looks tight` cue for the local-NLP recovery — bare integer feasibility isn't enough since continuous nonconvexities remain.

## 2. Convex relaxation at a node

For each nonconvex term, build the McCormick (Lesson 16) or αBB underestimator over the *current* box. The relaxation is an LP (if all relaxations are linear) or convex NLP, solved cheaply at each node.

This is why **bound tightening** (Lesson 15) is so critical in MINLP — each tightening *retroactively* tightens every relaxation in the tree.

## 3. Branching

Continuous branching variables are picked by:

- **Most violated** — branch on the variable whose envelope inequality is most violated by the LP solution.
- **Longest edge** — bisect the longest box edge.
- **Pseudocost-style** (recent literature).

Where to split (golden ratio? mid-point?): typically at the LP solution value or midpoint, with safeguards.

## 4. Convergence and termination

Spatial B&B is **finite for ε-optimality** but not finite in exact arithmetic for general nonconvex MINLPs. Convergence is by gap squeezing: every infinite branching sequence must shrink some box dimension to zero, eventually making the relaxation exact at the limit point {cite:p}`HorstTuy1996,Floudas1999`.

Practical termination: gap < ε (relative or absolute), or node/time limit.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model, sin, cos

# Nonconvex MINLP: pump network design (toy)
m = Model("pump_net")
x = m.continuous("x", lb=0, ub=10)     # flow
y = m.continuous("y", lb=0, ub=10)     # pressure
m.subject_to(x * y >= 6)                  # work
m.subject_to(0.5 * x**2 + y <= 12)        # head loss
m.minimize(0.1 * x**3 + y)                    # power cost

r = m.solve()
print(f"global: f* = {r.objective:.6f} at x = {r.x}")
print(f"closed gap to: {r.gap:.2e}, nodes = {r.node_count}")


## 5. When spatial B&B fails

- **Loose bounds** → loose envelopes → loose relaxations → no pruning. Fix: bound tightening, OBBT, branching on bounds.
- **Symmetry** → many equivalent nodes. Fix: lex breaking (Lesson 29).
- **High dimension** → curse of dimensionality on box partitioning. Fix: structure (Benders, Lagrangian) when applicable.

## References
{cite:p}`Smith1999,Tawarmalani2005,Belotti2009,Belotti2013,Floudas1999,HorstTuy1996`.